In [1]:
import time
import cv2
import numpy as np
import pygame
import torch as th
import robomimic.utils.file_utils as FileUtils
from collections import deque
import threading
import keyboard
from stable_baselines3.common.atari_wrappers import WarpFrame
from notebooks.resident4 import RE4Env
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import DummyVecEnv

# ===== CONFIGURAÇÃO DO MODELO =====
MODEL_PATH = r"d:/projects/robomimic/robomimic/logs/resident_evil/bc_transformer/test/20260326192243/last_bak.pth"
OBS_HORIZON = 24      # Quantos frames o Transformer olha (Context Length)
FRAME_SKIP = 4       # O intervalo de captura que você usou no dataset
ACTION_THRESHOLD_MOVE = 0.5
ACTION_THRESHOLD_BTN = 0.25

# ===== GLOBAIS =====
ai_queue = deque(maxlen=2)
latest_img = None
lock = threading.Lock()
running = True
could_use_ai = False

# ===== LOAD BC-TRANSFORMER =====
ckpt_dict = FileUtils.load_dict_from_checkpoint(MODEL_PATH)
device = "cuda"
policy, _ = FileUtils.policy_from_checkpoint(ckpt_dict=ckpt_dict, device=device)

def ai_worker():
    global latest_img, ai_queue, running, could_use_ai
    local_buffer = deque(maxlen=OBS_HORIZON)
    frame_count = 0
    
    while running:
        if could_use_ai and latest_img is not None:
            if frame_count % FRAME_SKIP == 0:
                # 1. Pre-processamento: Redimensiona e Transpose (C, H, W)
                # A ResNet18 treinada no Robomimic exige canais ANTES da altura/largura
                img = cv2.resize(latest_img, (128, 128))
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).transpose(2, 0, 1)
                local_buffer.append(img.astype(np.float32) / 255.0) # Normaliza aqui
                
                if len(local_buffer) == OBS_HORIZON:
                    # 2. Criar Tensor [Batch=1, Time=8, Channel=3, H=128, W=128]
                    obs_arr = np.array(list(local_buffer))
                    obs_tensor = th.from_numpy(obs_arr[None, ...]).to(device)
                    
                    # 3. Testar as duas chaves que o Robomimic costuma usar
                    # Tente primeiro 'image', se falhar o except tenta 'obs/image'
                    obs_dict = {"image": obs_tensor}
                    
                    try:
                        with th.no_grad():
                            # Chamada direta na rede de política (pula wrappers de validação)
                            # Isso costuma resolver o erro de "code should never reach this point"
                            action_tensor = policy.policy.nets["policy"](obs_dict)
                            
                            action_raw = action_tensor.cpu().numpy()[0]
                            
                            # Se o Transformer previu uma sequência, pegamos o frame atual [0]
                            if action_raw.ndim > 1:
                                action_raw = action_raw[0]
                                
                            with lock:
                                ai_queue.append(action_raw)
                                
                    except Exception as e:
                        # Se falhar com "image", tenta com "obs/image"
                        try:
                            action_tensor = policy.policy.nets["policy"]({"obs/image": obs_tensor})
                            action_raw = action_tensor.cpu().numpy()[0]
                            if action_raw.ndim > 1: action_raw = action_raw[0]
                            with lock: ai_queue.append(action_raw)
                        except Exception as e2:
                            print(f"❌ Erro Crítico de Arquitetura: {e2}")
            
            frame_count += 1
        else:
            frame_count = 0
            local_buffer.clear()
            
        time.sleep(0.001)

threading.Thread(target=ai_worker, daemon=True).start()

# ===== ENV & PYGAME SETUP =====
env = make_vec_env(lambda: WarpFrame(RE4Env(), 128, 128), n_envs=1)
pygame.init()
window = pygame.display.set_mode((640*3, 480*3))
clock = pygame.time.Clock()

last_action = np.zeros(18, dtype=np.int8)

try:
    while running:
        pygame.event.pump()
        if keyboard.is_pressed('esc'): running = False
        
        # Ativar/Desativar IA (Tecla K)
        if keyboard.is_pressed('k'):
            could_use_ai = not could_use_ai
            with lock: ai_queue.clear()
            print(f"MODO: {'🤖 AI' if could_use_ai else '👤 MANUAL'}")
            time.sleep(0.5)

        action = np.zeros(18, dtype=np.int8)

        # Botões manuais de emergência (sempre funcionam)
        if keyboard.is_pressed('i'): action[4] = 1 # Ex: Atirar
        if keyboard.is_pressed('o'): action[5] = 1 # Ex: Mirar
        if keyboard.is_pressed('p'): action[6] = 1 # Ex: Recarregar/X

        if not could_use_ai:
            if keyboard.is_pressed('up'):    action[0] = 1
            if keyboard.is_pressed('down'):  action[1] = 1
            if keyboard.is_pressed('left'):  action[2] = 1
            if keyboard.is_pressed('right'): action[3] = 1
        else:
            with lock:
                if len(ai_queue) > 0:
                    val = ai_queue.popleft()
                    # Aplica Thresholds (Movimento vs Botões)
                    action[0:4] = (val[0:4] > ACTION_THRESHOLD_MOVE).astype(np.int8)
                    action[4:] = (val[4:] > ACTION_THRESHOLD_BTN).astype(np.int8)
                    last_action = action
                else:
                    action = last_action

        # Aplica a ação no Resident Evil
        try:
            env.step(action[None, :])
        except:
            env.reset()

        # Captura e Renderização
        img = env.envs[0].env.env.render()
        if img is not None:
            latest_img = img.copy()
            img_rgb = cv2.cvtColor(cv2.resize(img, (640*3, 480*3)), cv2.COLOR_BGR2RGB)
            surface = pygame.image.frombuffer(img_rgb.tobytes(), (img_rgb.shape[1], img_rgb.shape[0]), "RGB")
            window.blit(surface, (0, 0))
            pygame.display.flip()

        clock.tick(30) # Mantém 30 FPS estáveis

finally:
    running = False
    pygame.quit()
    env.close()

pygame 2.6.1 (SDL 2.28.4, Python 3.8.0)
Hello from the pygame community. https://www.pygame.org/contribute.html


D:\anaconda3\envs\robomimic_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ROBOMIMIC WARNING(
    No private macro file found!
    It is recommended to use a private macro file
    To setup, run: python d:\projects\robomimic\robomimic/scripts/setup_macros.py
)

============= Initialized Observation Utils with Obs Spec =============

using obs modality: low_dim with keys: []
using obs modality: rgb with keys: ['image']
using obs modality: depth with keys: []
using obs modality: scan with keys: []
Created GPT_Backbone model with number of parameters: 18906112
6227978
Window founded HWND: 6227978.
MODO: 🤖 AI


D:\anaconda3\envs\robomimic_venv\lib\site-packages\torch\functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\TensorShape.cpp:3610.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


MODO: 👤 MANUAL
MODO: 🤖 AI


error: (1400, 'GetClientRect', 'O identificador da janela é inválido.')